# Set Operators - SQL

In [1]:
import pandas as pd
import numpy as np

In [2]:
df_customers = pd.read_csv('data/sales_customers.csv')
df_employees = pd.read_csv('data/sales_employees.csv')
df_orders = pd.read_csv('data/sales_orders.csv')
df_orderarchive = pd.read_csv('data/sales_ordersarchive.csv')
df_products = pd.read_csv('data/sales_products.csv')

### Rules Of Set Operators

- 1 Rule: ORDER BY can be userd only once
- 2 Rule: Same Number of Columns
- 3 Rule: Matching Data Types
- 4 Rule: Same Order of Columns
- 5 Rule: First Query Controls Aliases
- 6 Rule: Mapping Correct Columns

### UNION

- Returns all district rows from both queries
- Removes duplicate rows from the result

![Union](Pictures/Union.png)

### SQL TASK

#### Combine the data from employees and customers into one table

```SQL
SELECT
    firstname,
    lastname
FROM sales.customers

UNION

SELECT
    firstname,
    lastname
FROM sales.employees
```

In [5]:
df_cust_sub = df_customers[['firstname','lastname']]
df_emp_sub = df_employees[['firstname','lastname']]

df_union = pd.concat([df_cust_sub, df_emp_sub], ignore_index=True).drop_duplicates()

df_union

,firstname,lastname
0,Jossef,Goldberg
1,Kevin,Brown
2,Mary,NaN
3,Mark,Schwarz
4,Anna,Adams
5,Frank,Lee
8,Michael,Ray
9,Carol,Baker


### UNION ALL

#### Returns all rows from both queries, including duplicates

![Union All](Pictures/Union_All.png)

### SQL TASK

#### Combine the data from employees and customers into one table, including duplicates

```SQL
SELECT
    firstname,
    lastname
FROM sales.customers

UNION ALL

SELECT
    firstname,
    lastname
FROM sales.employees;
```

In [6]:
df_cust_sub = df_customers[['firstname','lastname']]
df_emp_sub = df_employees[['firstname','lastname']]

df_union = pd.concat([df_cust_sub, df_emp_sub], ignore_index=True)

df_union

,firstname,lastname
0,Jossef,Goldberg
1,Kevin,Brown
2,Mary,NaN
3,Mark,Schwarz
4,Anna,Adams
5,Frank,Lee
6,Kevin,Brown
7,Mary,NaN
8,Michael,Ray
9,Carol,Baker


### EXCEPT

#### - Returns all distinct rows from the first query that are not found in the second query
#### - It is the only one where the order of queries affect the final result

![Except](Pictures/Except.png)

### TASK SQL
#### Find employees who are not cutomers at the same time

```SQL
SELECT
    firstname,
    lastname
FROM sales.customers

EXCEPT

SELECT
    firstname,
    lastname
FROM sales.employees;
```

In [7]:
customers = df_customers[['firstname','lastname']]
employees = df_employees[['firstname','lastname']]

df_except = customers[~customers.apply(tuple, axis=1).isin(employees.apply(tuple, axis=1))]

df_except = df_except.drop_duplicates()

df_except

,firstname,lastname
0,Jossef,Goldberg
3,Mark,Schwarz
4,Anna,Adams


![operators](Pictures/operators_python.png)

### INTERSECT

#### Returns only the rows that are common in both queries

![Intersect](Pictures/Intersect.png)

### SQL TASK

#### Find employees who are also customers

```SQL
SELECT
    firstname,
    lastname
FROM sales.customers

INTERSECT

SELECT
    firstname,
    lastname
FROM sales.employees;
```

In [9]:
df_intersect = df_customers.merge(df_employees, on=['firstname','lastname'], how='inner')

df_intersect = df_intersect[['firstname','lastname']].drop_duplicates()

df_intersect

,firstname,lastname
0,Kevin,Brown
1,Mary,NaN


In [13]:
intersect_mask = df_customers.apply(tuple, axis=1).isin(df_employees.apply(tuple, axis=1))

# Así el merge no duplicará filas si un empleado aparece varias veces
df_intersect = df_customers.merge(
    df_employees[['firstname', 'lastname']].drop_duplicates(), 
    on=['firstname', 'lastname'], 
    how='inner'
)

df_intersect

,customerid,firstname,lastname,country,score
0,2,Kevin,Brown,USA,900.0
1,3,Mary,NaN,USA,750.0


### COMBINE INFORMATION

#### Combine similar information before analyzing the data

### SQL TASK

#### Orders are stored in separate tables (Orders and OrdersArchvie)
#### Combine all orders into one report without duplicates

```SQL
SELECT
    'orders' AS SourceTalbe,
    orderid,
    productid,
    customerid,
    salespersonid,
    orderdate,
    shipdate,
    orderstatus,
    shipaddress,
    billaddress,
    quantity,
    sales,
    creationtime
FROM sales.orders
UNION
SELECT
    'OrdersArchive' AS SourceTable,
    orderid,
    productid,
    customerid,
    salespersonid,
    orderdate,
    shipdate,
    orderstatus,
    shipaddress,
    billaddress,
    quantity,
    sales,
    creationtime
FROM sales.ordersarchive
ORDER BY orderid;
```

### DELTA DETECTION

#### Identifying the differences or changes (delta) between two batches of data